# IR System — Embedding Retrieval
**Step 6:** Dense retrieval using sentence-transformers (all-MiniLM-L6-v2).

⚠️ Use **GPU runtime** for this notebook: Runtime → Change runtime type → T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/ir_system_data'
import os, sys

if not os.path.exists('/content/ir-system'):
    !git clone https://github.com/ghazal-mohammad/ir-system.git /content/ir-system
else:
    !cd /content/ir-system && git pull
sys.path.insert(0, '/content/ir-system')

!pip install sentence-transformers==2.7.0 -q
print('ready')

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
import json
from services.embedding_service import load_model, build_doc_embeddings, save_embeddings

print('Loading model...')
model = load_model()
print('Model loaded')

# Load processed docs
print('Loading CT2021 docs...')
with open(f'{SAVE_DIR}/ct2021_docs_processed.json') as f:
    docs1 = json.load(f)
doc_ids1 = list(docs1.keys())
doc_texts1 = list(docs1.values())
print(f'CT2021: {len(doc_ids1):,} docs')

In [ ]:
# Encode CT2021 — takes ~30-40 min on GPU for 375K docs
doc_ids1_out, emb1 = build_doc_embeddings(doc_ids1, doc_texts1, model, batch_size=256)
print(f'Embeddings shape: {emb1.shape}')
save_embeddings(doc_ids1_out, emb1, f'{SAVE_DIR}/ct2021')
print('CT2021 embeddings saved')

In [ ]:
# Load MSMARCO docs
print('Loading MSMARCO docs...')
with open(f'{SAVE_DIR}/msmarco_docs_processed.json') as f:
    docs2 = json.load(f)
doc_ids2 = list(docs2.keys())
doc_texts2 = list(docs2.values())
print(f'MSMARCO: {len(doc_ids2):,} docs')

# Encode MSMARCO — takes ~20-25 min on GPU for 500K docs
doc_ids2_out, emb2 = build_doc_embeddings(doc_ids2, doc_texts2, model, batch_size=256)
print(f'Embeddings shape: {emb2.shape}')
save_embeddings(doc_ids2_out, emb2, f'{SAVE_DIR}/msmarco')
print('MSMARCO embeddings saved')

In [ ]:
# Test retrieval on sample query
import ir_datasets
from services.embedding_service import load_embeddings, retrieve_embedding

doc_ids1_loaded, emb1_loaded = load_embeddings(f'{SAVE_DIR}/ct2021')

ds1 = ir_datasets.load('clinicaltrials/2021/trec-ct-2021')
queries1 = {q.query_id: q.text for q in ds1.queries_iter()}

sample_qid = list(queries1.keys())[0]
sample_query = queries1[sample_qid]

results = retrieve_embedding(sample_query, model, doc_ids1_loaded, emb1_loaded, top_k=10)
print(f'Query: "{sample_query}"')
print('Top 10 Embedding results:')
for r in results:
    print(f'  rank {r["rank"]}: {r["doc_id"]} (score={r["score"]})')

In [ ]:
# Full retrieval for all queries — CT2021
import json
from services.embedding_service import load_embeddings, retrieve_embedding

doc_ids1_loaded, emb1_loaded = load_embeddings(f'{SAVE_DIR}/ct2021')
doc_ids2_loaded, emb2_loaded = load_embeddings(f'{SAVE_DIR}/msmarco')

ds1 = ir_datasets.load('clinicaltrials/2021/trec-ct-2021')
queries1 = {q.query_id: q.text for q in ds1.queries_iter()}
ds2 = ir_datasets.load('msmarco-passage/trec-dl-2019')
queries2 = {q.query_id: q.text for q in ds2.queries_iter()}

print('Running embedding retrieval on CT2021...')
all_results1 = {}
for qid, qtext in queries1.items():
    all_results1[qid] = retrieve_embedding(qtext, model, doc_ids1_loaded, emb1_loaded, top_k=1000)
with open(f'{SAVE_DIR}/ct2021_embedding_results.json', 'w') as f:
    json.dump(all_results1, f)
print(f'CT2021 done: {len(all_results1)} queries')

print('Running embedding retrieval on MSMARCO...')
all_results2 = {}
for qid, qtext in queries2.items():
    all_results2[qid] = retrieve_embedding(qtext, model, doc_ids2_loaded, emb2_loaded, top_k=1000)
with open(f'{SAVE_DIR}/msmarco_embedding_results.json', 'w') as f:
    json.dump(all_results2, f)
print(f'MSMARCO done: {len(all_results2)} queries')

print('\n=== Embedding Retrieval Complete ===')
print('Next: 07_retrieval_hybrid.ipynb')